# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedAmmarSharif/Flyrank-ML-Internship-Work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")
print("Dataset shape:", df.shape)

stale_bins = [-1, 30, 90, 180, 365, float("inf")]
stale_labels = ["0-30", "31-90", "91-180", "181-365", "365+"]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=stale_bins,
    labels=stale_labels
)

staleness_table = df.groupby("staleness_bucket", observed=False).agg(n=("content_id", "size"))

print("Staleness signal")
display(staleness_table)

volume_bins = [-1, 100, 500, 2000, 10000, float("inf")]
volume_labels = ["0-100", "101-500", "501-2k", "2k-10k", "10k+"]

df["volume_bucket"] = pd.cut(
df["impressions_90d"],
    bins=volume_bins,
    labels=volume_labels
)

visibility_table = df.groupby("volume_bucket", observed=False).agg(n=("content_id", "size"), median_ctr=("ctr", "median"))

print("Visibility signal")
display(visibility_table)

Dataset shape: (30000, 44)
Staleness signal


,n
staleness_bucket,
0-30,20480
31-90,175
91-180,9171
181-365,169
365+,5


Visibility signal


,n,median_ctr
volume_bucket,,
0-100,8006,0.00
101-500,5279,0.00
501-2k,6502,0.12
2k-10k,6611,0.19
10k+,3602,0.23


1. My rule and its reason codes
Signal check 1: Staleness
Verdict: MIXED

Most pages were updated within the last 30 days (20,480 pages). Only 174 pages have not been updated for more than 180 days. Therefore, staleness alone may not be a strong enough reason to recommend a refresh.

Signal check 2: Visibility
Verdict: CONFIRMED

Median CTR generally increases across higher impression buckets, from 0.00 in the lowest buckets to 0.23 in the highest bucket. This supports using visibility as a prioritization signal, but it does not prove that refreshing a page will improve performance.

Initial rule idea
Prioritize pages with meaningful search visibility and an older last-update date for human review.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


df["baseline_score"] = (
    (df["days_since_last_update"] >= 90).astype(int)
    * (df["impressions_90d"] >= 500).astype(int)
    * df["impressions_90d"]
)

df["reason_code"] = np.where(
    (df["days_since_last_update"] >= 90) &
    (df["impressions_90d"] >= 500),
    "VISIBLE_AND_STALE",
    "NOT_PRIORITIZED"
)

df["recommended_action"] = np.where(
    df["baseline_score"] > 0,
    "REVIEW_FOR_REFRESH",
    "MONITOR_OR_LEAVE"
)

baseline_queue = df.sort_values(
    "baseline_score",
    ascending=False
).copy()

baseline_queue["rank"] = range(1, len(baseline_queue) + 1)

output_path = Path("work/outputs")
output_path.mkdir(parents=True, exist_ok=True)

baseline_queue.to_csv(
    output_path / "baseline_action_score.csv",
    index=False
)

print("Baseline queue created!")
print("Total pages:", len(baseline_queue))

display(baseline_queue[["rank", "content_id", "days_since_last_update", "impressions_90d", "baseline_score", "reason_code", "recommended_action"]].head(10).reset_index(drop=True))

Baseline queue created!
Total pages: 30000


,rank,content_id,days_since_last_update,impressions_90d,baseline_score,reason_code,recommended_action
0,1,content_5fe46e04994d,104,517715,517715,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
1,2,content_2dba2b1f9536,104,443434,443434,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
2,3,content_2c2606c5d176,104,347399,347399,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
3,4,content_cb112fce36be,104,309910,309910,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
4,5,content_9532f197bbc8,104,309192,309192,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
5,6,content_36ff89c8214e,104,295097,295097,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
6,7,content_b28d1efd668f,104,286608,286608,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
7,8,content_813e88069237,104,233561,233561,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
8,9,content_c21024970297,104,211366,211366,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
9,10,content_c8e9d6ab9013,104,208678,208678,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH


In [14]:
review_cols = [
    "rank",
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "word_count",
    "engagement_rate",
    "scroll_rate",
    "baseline_score",
    "reason_code",
    "recommended_action"
]

review_cols = [col for col in review_cols if col in baseline_queue.columns]

top_10_review = baseline_queue[review_cols].head(10)

display(top_10_review.reset_index(drop=True))

,rank,content_id,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,word_count,engagement_rate,scroll_rate,baseline_score,reason_code,recommended_action
0,1,content_5fe46e04994d,104,517715,741,0.14,4.2,537,NaN,4.23,26.90,517715,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
1,2,content_2dba2b1f9536,104,443434,910,0.21,27.9,299,7676.0,2.73,7.77,443434,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
2,3,content_2c2606c5d176,104,347399,1854,0.53,4.2,362,NaN,1.30,3.28,347399,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
3,4,content_cb112fce36be,104,309910,492,0.16,5.6,126,2761.0,2.08,2.95,309910,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
4,5,content_9532f197bbc8,104,309192,2689,0.87,2.0,445,NaN,8.01,28.75,309192,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
5,6,content_36ff89c8214e,104,295097,154,0.05,7.3,144,NaN,1.68,4.48,295097,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
6,7,content_b28d1efd668f,104,286608,169,0.06,26.2,153,6901.0,3.83,6.57,286608,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
7,8,content_813e88069237,104,233561,129,0.06,26.2,153,4610.0,1.37,10.37,233561,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
8,9,content_c21024970297,104,211366,870,0.41,5.1,126,2874.0,1.49,3.10,211366,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH
9,10,content_c8e9d6ab9013,104,208678,0,0.00,9.7,362,NaN,0.00,0.00,208678,VISIBLE_AND_STALE,REVIEW_FOR_REFRESH


## 3. Top-10 review

1. **Rank 1 — REVIEW_FOR_REFRESH:** The page has 517,715 impressions and has not been updated for 104 days. Its CTR is 0.14%. The recommendation could be wrong if the page is already performing well or does not need updated information.

2. **Rank 2 — REVIEW_FOR_REFRESH:** The page has 443,434 impressions and is 104 days old since its last update. Its CTR is 0.21%. The recommendation could be wrong if its content is still accurate and users are engaging successfully.

3. **Rank 3 — REVIEW_FOR_REFRESH:** The page has 347,399 impressions and a CTR of 0.53%. The recommendation could be wrong if its current content is already relevant and no meaningful improvement is needed.

4. **Rank 4 — REVIEW_FOR_REFRESH:** The page has 309,910 impressions and a CTR of 0.16%. A review may help identify presentation or content issues. The recommendation could be wrong if the page is already meeting its intended goals.

5. **Rank 5 — REVIEW_FOR_REFRESH:** The page has 309,192 impressions and a CTR of 0.87%. Its relatively high CTR may indicate that it is already attracting clicks. Refreshing it may be unnecessary.

6. **Rank 6 — REVIEW_FOR_REFRESH:** The page has 295,097 impressions but a CTR of only 0.05%. A human review could investigate its search presentation. The recommendation could be wrong if the low CTR is expected for its search intent.

7. **Rank 7 — REVIEW_FOR_REFRESH:** The page has 286,608 impressions and a CTR of 0.06%. Its low CTR makes it worth investigating. The recommendation could be wrong if its search position or query intent naturally results in fewer clicks.

8. **Rank 8 — REVIEW_FOR_REFRESH:** The page has 233,561 impressions and a CTR of 0.06%. A review could investigate whether its title and search presentation are effective. The recommendation could be wrong if its existing content is already suitable.

9. **Rank 9 — REVIEW_FOR_REFRESH:** The page has 211,366 impressions and a CTR of 0.41%. It has meaningful visibility and may deserve a review. The recommendation could be wrong if the page is already performing satisfactorily.

10. **Rank 10 — REVIEW_FOR_REFRESH:** The page has 208,678 impressions but zero recorded clicks and a CTR of 0.00%. This makes it worth investigating. The recommendation could be wrong if the zero-click result is due to its search intent or measurement limitations.

### Overall observation

All ten pages meet the same staleness and visibility conditions. However, their CTR values vary considerably. This shows that the baseline rule is useful for prioritizing human review, but it does not fully capture page performance or guarantee that a refresh is necessary.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

1. **Rank 5:** This page has a relatively high CTR of 0.87%. Although it is stale and highly visible, refreshing it may not be necessary because it already attracts clicks.

2. **Rank 10:** This page has zero recorded clicks and a CTR of 0.00%. The rule recommends a refresh, but the issue could be related to search intent, measurement limitations, or other factors that require investigation.

3. **Rank 1:** This page has the highest impressions but a CTR of only 0.14%. High visibility does not automatically mean that a content refresh is the best action.

### Lesson

The baseline rule uses only impressions and staleness. It can identify pages worth reviewing, but it cannot determine the exact cause of poor performance or guarantee that refreshing content will improve results.

In [16]:

rule_inputs = ["days_since_last_update", "impressions_90d"]

forbidden_inputs = ["trend_direction", "trend_pct"]

print("Rule inputs:", rule_inputs)
print("Forbidden label fields used in rule:", [col for col in forbidden_inputs if col in rule_inputs])

print("Score formula:")
print("(days_since_last_update >= 90) AND (impressions_90d >= 500)")
print("Score = impressions_90d when both conditions are true; otherwise 0")

Rule inputs: ['days_since_last_update', 'impressions_90d']
Forbidden label fields used in rule: []
Score formula:
(days_since_last_update >= 90) AND (impressions_90d >= 500)
Score = impressions_90d when both conditions are true; otherwise 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.